Dataset Loading
This project aims to predict whether an order item will be returned.

The datasets loaded include:

Orders and Transactions
Product Catalogue
Returns
Customer Purchase and Return History
Marketplace Seller Data
The target variable is Returned:

1 = Returned
0 = Not Returned

In [13]:
import pandas as pd

# Load datasets from your PC
orders = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table2_orders_and_transactions.csv"
)

products = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table3_product_catalogue.csv"
)

returns = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table4_returns.csv"
)

customers = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table5_customer_purchase_and_return_history.csv"
)

sellers = pd.read_csv(
    r"C:\Users\w7127363\Downloads\table6_marketplace_seller_data.csv"
)

# Display dataset dimensions
print("Orders:", orders.shape)
print("Products:", products.shape)
print("Returns:", returns.shape)
print("Customers:", customers.shape)
print("Sellers:", sellers.shape)

Orders: (48788, 15)
Products: (3209, 14)
Returns: (7657, 12)
Customers: (48643, 13)
Sellers: (48643, 12)


In [14]:
returned_items = set(
    returns["Order_Item_ID"]
)

orders["Returned"] = (
    orders["Order_Item_ID"]
    .isin(returned_items)
    .astype(int)
)

orders["Returned"].value_counts()

Returned
0    41139
1     7649
Name: count, dtype: int64

Dataset Preprocessing
The datasets are merged using their shared identifiers to create a single modelling dataset.

Data preprocessing includes:

Creating the target variable (Returned)
Joining customer, seller and product information
Handling missing values
Removing duplicate records
Encoding categorical variables
Splitting the data into training, validation and testing datasets
Customer-level splitting is used to prevent data leakage by ensuring that the same customer does not appear in both training and testing data.

In [15]:
print("Orders duplicates:",
      orders.duplicated(subset=["Order_Item_ID"]).sum())

print("Products duplicates:",
      products.duplicated(subset=["SKU"]).sum())

print("Returns duplicates:",
      returns.duplicated(subset=["Return_ID"]).sum())

Orders duplicates: 145
Products duplicates: 9
Returns duplicates: 22


In [16]:
orders = orders.drop_duplicates(
    subset=["Order_Item_ID"]
)

products = products.drop_duplicates(
    subset=["SKU"]
)

returns = returns.drop_duplicates(
    subset=["Return_ID"]
)

In [17]:
df = orders.merge(
    customers,
    on="Order_Item_ID",
    how="left"
)

df = df.merge(
    sellers,
    on="Order_Item_ID",
    how="left"
)

df = df.merge(
    products,
    on="SKU",
    how="left"
)

print(df.shape)

(48643, 52)


In [18]:
missing_values = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

print(
    missing_values[
        missing_values > 0
    ]
)


Size                      33522
Promotion_Type            29886
Average_Order_Value       16613
Days_Since_Last_Order     16613
Historical_Return_Rate     8878
Colour                     7753
Product_Attributes         5668
Listing_Age                5212
Average_Rating             4375
Review_Count               4138
Product_Subcategory        3586
Brand                      2523
Seller_Return_Rate          419
Average_Shipping_Time       419
Late_Shipment_Rate          419
dtype: int64


In [19]:
for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].fillna(
        df[col].median()
    )

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna(
        "Unknown"
    )

In [20]:
from sklearn.model_selection import train_test_split

customer_ids = df["Customer_ID_x"].unique()

train_customers, temp_customers = train_test_split(
    customer_ids,
    test_size=0.30,
    random_state=42
)

val_customers, test_customers = train_test_split(
    temp_customers,
    test_size=0.50,
    random_state=42
)

train_df = df[
    df["Customer_ID_x"].isin(train_customers)
]

val_df = df[
    df["Customer_ID_x"].isin(val_customers)
]

test_df = df[
    df["Customer_ID_x"].isin(test_customers)
]

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (33851, 52)
Validation: (7384, 52)
Testing: (7408, 52)


Model Training
A Random Forest Classifier was selected because it performs well on structured tabular datasets and can model complex relationships between variables.

Hyperparameter choices:

n_estimators = 200
max_depth = 15
class_weight = balanced
random_state = 42
These values were selected to improve predictive performance while reducing the risk of overfitting. The balanced class weighting also helps address any imbalance between returned and non-returned orders.

In [21]:
drop_columns = [
    "Returned",
    "Order_Item_ID",
    "Order_ID_x",
    "Order_ID_y",
    "Order_ID",
    "Customer_ID_x",
    "Customer_ID_y"
]

X_train = train_df.drop(
    columns=drop_columns,
    errors="ignore"
)

y_train = train_df["Returned"]

X_val = val_df.drop(
    columns=drop_columns,
    errors="ignore"
)

y_val = val_df["Returned"]

X_test = test_df.drop(
    columns=drop_columns,
    errors="ignore"
)

y_test = test_df["Returned"]

In [22]:
X_train = pd.get_dummies(
    X_train,
    drop_first=True
)

X_val = pd.get_dummies(
    X_val,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    drop_first=True
)

X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(33851, 59969)
(7384, 59969)
(7408, 59969)


In [23]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    class_weight="balanced",
    random_state=42
)

model.fit(
    X_train,
    y_train
)

print("Model training complete.")

Model training complete.


Model Validation
The trained model is evaluated using the unseen test dataset.

The following performance metrics are used:

Accuracy
Precision
Recall
F1-Score
These metrics provide insight into how well the model predicts returned and non-returned order items.

In [24]:
predictions = model.predict(X_test)

In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(
    y_test,
    predictions
)

precision = precision_score(
    y_test,
    predictions
)

recall = recall_score(
    y_test,
    predictions
)

f1 = f1_score(
    y_test,
    predictions
)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

Accuracy: 0.7204
Precision: 0.2745
Recall: 0.468
F1 Score: 0.3461


In [26]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predictions
    )
)

              precision    recall  f1-score   support

           0       0.88      0.77      0.82      6237
           1       0.27      0.47      0.35      1171

    accuracy                           0.72      7408
   macro avg       0.58      0.62      0.58      7408
weighted avg       0.79      0.72      0.75      7408



In [27]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    predictions
)

print(cm)

[[4789 1448]
 [ 623  548]]
